<a href="https://colab.research.google.com/github/arjunnamburi/ml_death_overs_ipl/blob/main/notebooks/post_freeze_checks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ===== T0. Setup and load (COPY-PASTE) =====
import json, os, time, datetime, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/IPL PROJECT ALL DOCS/data/'

df = pd.read_parquet(DATA + 'ipl_features.parquet')
with open(DATA + 'feature_list.json') as f: meta = json.load(f)
with open(DATA + 'freeze.json') as f:       FZ = json.load(f)
with open(DATA + 'calibration.json') as f:  CALJ = json.load(f)

FEATURES, CAT_FEATURES, CLASSES = meta['features'], meta['categorical'], meta['classes']
HIST = sum(meta['hist_features'].values(), [])
NUM = [c for c in FEATURES if c not in CAT_FEATURES]
NUM_H = NUM + HIST                        # B2-hist numeric inputs (82)
FEAT_HIST = list(FEATURES) + HIST         # X-hist inputs (87)
K = len(CLASSES)

# legal balls of 2008-2026; rain-affected matches have split 'rain' and stay out
d = df[(df['y'] >= 0) & df['split'].isin(['train', 'val', 'test'])].copy()
d['y'] = d['y'].astype(int)
for c in CAT_FEATURES:
    if not isinstance(d[c].dtype, pd.CategoricalDtype):
        d[c] = d[c].astype('category')

TR    = (d['split'] == 'train').to_numpy()     # 2008-2024: used only to pick X-hist's tree count
VA    = (d['split'] == 'val').to_numpy()       # 2025
FIT   = TR | VA                                # 2008-2025: every final model is refit here
TE    = (d['split'] == 'test').to_numpy()      # 2026: scored once, in T7
DEATH = (d['match_phase'] == 'death').to_numpy()

Mounted at /content/drive


In [3]:
# ===== A0. d with rain matches + extra masks (COPY-PASTE) =====
d = df[(df['y'] >= 0) & df['split'].isin(['train', 'val', 'test', 'rain'])].copy()
d['y'] = d['y'].astype(int)
for c in CAT_FEATURES:
    if not isinstance(d[c].dtype, pd.CategoricalDtype):
        d[c] = d[c].astype('category')

TR    = (d['split'] == 'train').to_numpy()
VA    = (d['split'] == 'val').to_numpy()
FIT   = TR | VA
TE    = (d['split'] == 'test').to_numpy()
RAIN  = (d['split'] == 'rain').to_numpy()
DEATH = (d['match_phase'] == 'death').to_numpy()
IT    = TR & (d['year'] <= 2023).to_numpy()      # inner train, as in X0
IV    = TR & (d['year'] == 2024).to_numpy()      # inner val (early stopping for 2025 fits)
MATCH = d['match_id'].to_numpy()

print(d['split'].value_counts().to_dict())       # expect train 244269, val 15809, test 16282, rain 8101
print('rain matches by year:', d.loc[RAIN].groupby('year')['match_id'].nunique().to_dict())

{'train': 244269, 'test': 16282, 'val': 15809, 'rain': 8101}
rain matches by year: {2008: 5, 2009: 5, 2010: 1, 2011: 5, 2012: 1, 2013: 2, 2014: 3, 2015: 6, 2016: 4, 2017: 1, 2018: 3, 2019: 2, 2023: 3, 2024: 1, 2025: 5, 2026: 3}


In [4]:
# ===== T2. Helpers (COPY-PASTE: copies of models.ipynb functions) =====
EPS = 1e-15
def per_ball_logloss(P, y):
    return -np.log(np.clip(P[np.arange(len(y)), y], EPS, 1))

def per_ball_brier(P, y):
    O = np.zeros_like(P); O[np.arange(len(y)), y] = 1
    return ((P - O) ** 2).sum(axis=1)

def slice_paired(la, lb, mask, match, n_boot=2000, z_bonf=3.254, seed=0):
    """Mean of (la - lb) over the slice, 95% match-bootstrap interval, SE, Bonferroni flag. lb = 0 gives one model's interval."""
    diff, m = (la - lb)[mask], match[mask]
    codes, _ = pd.factorize(m)
    s, c = np.bincount(codes, weights=diff), np.bincount(codes)
    idx = np.random.default_rng(seed).integers(0, len(s), size=(n_boot, len(s)))
    boot = s[idx].sum(axis=1) / c[idx].sum(axis=1)
    mean, se = diff.mean(), boot.std()
    low, high = np.percentile(boot, [2.5, 97.5])
    return mean, low, high, se, abs(mean) > z_bonf * se

# --- B0 / B1 ---
def fit_freq(y): return np.bincount(y, minlength=K) / len(y)

WKT_BAND = pd.cut(d['c_team_wickets'], bins=[-1, 2, 5, 7, 9], labels=['0-2', '3-5', '6-7', '8-9'])
d['cell'] = d['match_phase'].astype(str) + ' | ' + WKT_BAND.astype(str) + ' wkts | inn ' + d['innings'].astype(str)
ALL_CELLS = [f'{ph} | {w} wkts | inn {i}' for ph in ['powerplay', 'middle', 'death']
             for w in ['0-2', '3-5', '6-7', '8-9'] for i in [1, 2]]

def fit_b1(rows, k=100):
    counts = (pd.crosstab(d.loc[rows, 'cell'], d.loc[rows, 'y'])
                .reindex(index=ALL_CELLS, columns=range(K), fill_value=0))
    prior = fit_freq(d.loc[rows, 'y'].values)
    n = counts.sum(axis=1).values[:, None]
    probs = np.where(n + k > 0, (counts.values + k * prior) / np.maximum(n + k, 1), prior)
    return pd.DataFrame(probs, index=ALL_CELLS, columns=CLASSES), counts.sum(axis=1)

def predict_b1(table, rows): return table.loc[d.loc[rows, 'cell']].values

# --- B2-hist ---
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning

def missing_groups(rows, cols):
    miss = d.loc[rows, cols].isna()
    miss = miss.loc[:, miss.any()]
    groups = {}
    for c in miss.columns:
        groups.setdefault(miss[c].values.tobytes(), []).append(c)
    return list(groups.values())

def fit_prep(rows, num, cat):
    X = d.loc[rows, num]
    lo, hi = X.quantile(0.005), X.quantile(0.995)
    few = X.nunique() <= 20
    lo[few], hi[few] = -np.inf, np.inf
    Xc = X.clip(lo, hi, axis=1)
    return {'num': num, 'cat': cat, 'lo': lo, 'hi': hi, 'mean': Xc.mean(), 'std': Xc.std().replace(0, 1),
            'groups': missing_groups(rows, num)}

def transform(prep, rows):
    X = d.loc[rows, prep['num']]
    Z = ((X.clip(prep['lo'], prep['hi'], axis=1) - prep['mean']) / prep['std']).fillna(0)
    flags = pd.DataFrame({'miss_' + g[0]: X[g[0]].isna().astype(float) for g in prep['groups']}, index=X.index)
    onehot = pd.get_dummies(d.loc[rows, prep['cat']], dtype=float)
    return pd.concat([Z, flags, onehot], axis=1)

def fit_lr(X, y, C=1.0):
    m = LogisticRegression(C=C, max_iter=3000)
    with warnings.catch_warnings():
        warnings.simplefilter('error', ConvergenceWarning)
        m.fit(X, y)
    return m

def to_logits(P, eps=1e-12): return np.log(np.clip(P, eps, 1))
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True); E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)
def apply_vector(P, a, b): return softmax(to_logits(P) * a + b)

# --- X-hist ---
import xgboost as xgb
XP = FZ['final_models']['X-hist']['params']
BEST = dict(max_depth=XP['max_depth'], min_child_weight=XP['min_child_weight'])
BASE_PARAMS = dict(objective='multi:softprob', eval_metric='mlogloss', tree_method='hist',
                   device='cuda', enable_categorical=True,
                   learning_rate=XP['learning_rate'], subsample=XP['subsample'],
                   colsample_bytree=XP['colsample_bytree'], random_state=0)

def fit_xgb_es(feats, fit_rows, stop_rows, **params):
    m = xgb.XGBClassifier(**BASE_PARAMS, **params, n_estimators=5000, early_stopping_rounds=50)
    m.fit(d.loc[fit_rows, feats], d.loc[fit_rows, 'y'],
          eval_set=[(d.loc[stop_rows, feats], d.loc[stop_rows, 'y'])], verbose=200)
    return m, m.best_iteration + 1, m.best_score

def fit_xgb_fixed(feats, rows, n_trees, **params):
    m = xgb.XGBClassifier(**BASE_PARAMS, **params, n_estimators=n_trees)
    m.fit(d.loc[rows, feats], d.loc[rows, 'y'], verbose=False)
    return m

print('xgboost', xgb.__version__, '| frozen X-hist settings:', BEST, XP)   # expect depth 4, mcw 100, lr .05, sub/col .8

xgboost 3.4.1 | frozen X-hist settings: {'max_depth': 4, 'min_child_weight': 100} {'max_depth': 4, 'min_child_weight': 100, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8}


In [5]:
# ===== A1. Feature groups (COPY-PASTE) =====
INMATCH, taken, G = list(FEATURES), set(), {}
def take(rule):
    cols = [c for c in INMATCH if c not in taken and rule(c)]
    taken.update(cols)
    return cols

G['delivery']     = take(lambda c: c in ('is_free_hit', 'prev_ball_outcome'))
G['chase']        = take(lambda c: any(k in c for k in ('required', 'need', 'pressure', 'runs_target')))
G['partnership']  = take(lambda c: 'partnership' in c or 'non_striker' in c or 'nonstriker' in c
                                   or c.startswith('ns_') or 'left_right' in c)
G['striker']      = take(lambda c: c.startswith('batter_') and c != 'batter_hand')
G['bowler_spell'] = take(lambda c: (c.startswith('bowler_') and c != 'bowler_cat')
                                   or 'spell' in c or 'since_last_bowled' in c)
G['momentum']     = take(lambda c: any(k in c for k in ('last6', 'last12', 'since', 'this_over')))
SITUATION = [c for c in INMATCH if c not in taken]
for k, v in meta['hist_features'].items():
    G['hist_' + k] = list(v)

for g, cols in G.items():
    print(f'{g:22s} {len(cols):2d}  {cols}')
print(f'{"SITUATION (kept)":22s} {len(SITUATION):2d}  {SITUATION}')
print('total:', sum(map(len, G.values())) + len(SITUATION))   # expect 87

delivery                2  ['is_free_hit', 'prev_ball_outcome']
chase                   6  ['runs_target', 'runs_needed', 'required_run_rate', 'pressure_diff', 'runs_needed_per_wicket', 'recent_pressure_diff']
partnership             7  ['non_striker_pos', 'partnership_runs', 'partnership_balls', 'non_striker_runs', 'non_striker_balls', 'non_striker_sr', 'is_left_right_pair']
striker                 5  ['batter_sr_innings', 'batter_runs_last6', 'batter_dots_last6', 'batter_dots_in_row', 'batter_bdry_run_pct']
bowler_spell            7  ['bowler_econ_match', 'overs_since_last_bowled', 'bowler_last_over_runs', 'spell_balls', 'spell_runs', 'spell_wkts', 'spell_dots']
momentum               12  ['team_runs_last6', 'team_runs_last12', 'team_wkts_last6', 'team_wkts_last12', 'team_dots_last6', 'team_dots_last12', 'team_bdry_last6', 'team_bdry_last12', 'team_extras_last12', 'balls_since_last_boundary', 'balls_since_last_wicket', 'runs_this_over']
hist_8_batter           9  ['bat_balls_3s', 'ba

In [6]:
# ===== A2. Drop-one-group refit (TYPE IT) =====
SEASON = {
    2025: dict(es_fit=IT, es_stop=IV, fit=TR,  score=VA),   # as in X3 (340 trees for the full model)
    2026: dict(es_fit=TR, es_stop=VA, fit=FIT, score=TE),   # as in T5 (301 trees for the full model)
}

def refit_without(drop, year):
    s = SEASON[year]
    feats = [c for c in FEAT_HIST if c not in set(drop)]
    _, n_trees, _ = fit_xgb_es(feats, s['es_fit'], s['es_stop'], **BEST)   # tree count, frozen way
    m = fit_xgb_fixed(feats, s['fit'], n_trees, **BEST)                    # refit with that count
    return m, n_trees, m.predict_proba(d.loc[s['score'], feats])

In [7]:
# ===== A3. Run all refits (COPY-PASTE, ~30-40 min on GPU) =====
YEARS = [2025, 2026]
FULL, ABL = {}, {}
for yr in YEARS:
    yv = d.loc[SEASON[yr]['score'], 'y'].to_numpy()
    t0 = time.time()
    FULL[yr] = refit_without([], yr)
    print(f'{yr} full X-hist: {FULL[yr][1]} trees | log-loss {per_ball_logloss(FULL[yr][2], yv).mean():.4f} '
          f'| {time.time() - t0:.0f} s')
    # expect 2025: 340 trees, 1.4727 | 2026: 301 trees, 1.4787 (GPU may move the 4th decimal by 1-2)
    for g in G:
        t0 = time.time()
        _, n, Pg = refit_without(G[g], yr)
        ABL[(yr, g)] = (n, Pg)
        print(f'   without {g:22s} {n:4d} trees | log-loss {per_ball_logloss(Pg, yv).mean():.4f} '
              f'| {time.time() - t0:.0f} s')

np.savez_compressed(DATA + 'ablation_preds.npz',
                    **{f'{yr}_full': FULL[yr][2] for yr in YEARS},
                    **{f'{yr}_{g}': Pg for (yr, g), (n, Pg) in ABL.items()})

[0]	validation_0-mlogloss:1.54249
[200]	validation_0-mlogloss:1.47430
[389]	validation_0-mlogloss:1.47348


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [05:16:03] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


2025 full X-hist: 340 trees | log-loss 1.4727 | 33 s
[0]	validation_0-mlogloss:1.54247
[200]	validation_0-mlogloss:1.47426
[400]	validation_0-mlogloss:1.47356
[476]	validation_0-mlogloss:1.47364
   without delivery                427 trees | log-loss 1.4740 | 40 s
[0]	validation_0-mlogloss:1.54234
[200]	validation_0-mlogloss:1.47540
[356]	validation_0-mlogloss:1.47443
   without chase                   307 trees | log-loss 1.4738 | 29 s
[0]	validation_0-mlogloss:1.54235
[200]	validation_0-mlogloss:1.47467
[400]	validation_0-mlogloss:1.47351
[467]	validation_0-mlogloss:1.47370
   without partnership             418 trees | log-loss 1.4734 | 36 s
[0]	validation_0-mlogloss:1.54239
[200]	validation_0-mlogloss:1.47492
[400]	validation_0-mlogloss:1.47411
[413]	validation_0-mlogloss:1.47411
   without striker                 364 trees | log-loss 1.4745 | 36 s
[0]	validation_0-mlogloss:1.54248
[200]	validation_0-mlogloss:1.47434
[390]	validation_0-mlogloss:1.47328
   without bowler_spell      

In [8]:
# ===== A4. Ablation table (COPY-PASTE) =====
from scipy.stats import norm
z_ab = norm.ppf(1 - 0.025 / (2 * len(G)))          # 11 groups x (all, death) per season
rows = []
for yr in YEARS:
    sc = SEASON[yr]['score']
    yv, mt, dth = d.loc[sc, 'y'].to_numpy(), MATCH[sc], DEATH[sc]
    L_full = per_ball_logloss(FULL[yr][2], yv)
    for g in G:
        n, Pg = ABL[(yr, g)]
        L_g = per_ball_logloss(Pg, yv)
        for balls, mask in [('all', np.ones(len(yv), bool)), ('death', dth)]:
            mean, lo, hi, se, bonf = slice_paired(L_g, L_full, mask, mt, z_bonf=z_ab)
            rows.append(dict(season=yr, group=g, n_feats=len(G[g]), trees=n, balls=balls,
                             loss_if_dropped=mean, low=lo, high=hi, bonferroni=bonf))
ab = pd.DataFrame(rows)
ab['verdict'] = np.select([ab['low'] > 0, ab['high'] < 0], ['group helps', 'group hurts'], "can't tell")
ab.to_csv(DATA + 'ablation_groups.csv', index=False)
print(f'Bonferroni z = {z_ab:.3f}')
print(ab.round(4).to_string(index=False))

Bonferroni z = 3.023
 season                group  n_feats  trees balls  loss_if_dropped     low   high  bonferroni     verdict
   2025             delivery        2    427   all           0.0013  0.0006 0.0021        True group helps
   2025             delivery        2    427 death           0.0007 -0.0013 0.0027       False  can't tell
   2025                chase        6    307   all           0.0011 -0.0000 0.0021       False  can't tell
   2025                chase        6    307 death           0.0029  0.0001 0.0056       False group helps
   2025          partnership        7    418   all           0.0007 -0.0002 0.0017       False  can't tell
   2025          partnership        7    418 death           0.0012 -0.0008 0.0032       False  can't tell
   2025              striker        5    364   all           0.0018  0.0012 0.0025        True group helps
   2025              striker        5    364 death           0.0025  0.0005 0.0046       False group helps
   2025         

In [9]:
# ===== A5. Rain-affected matches, descriptive (COPY-PASTE) =====
a26 = np.array(CALJ['B2-hist']['a_for_2026']); b26 = np.array(CALJ['B2-hist']['b_for_2026'])
rows = []
for yr, fit_rows in [(2025, TR), (2026, FIT)]:
    R = RAIN & (d['year'] == yr).to_numpy()
    if R.sum() == 0:
        print(yr, ': no rain-affected legal balls'); continue
    yv, mt, dth = d.loc[R, 'y'].to_numpy(), MATCH[R], DEATH[R]
    b1_tab, _ = fit_b1(fit_rows, k=100)
    PR = {'X-hist': FULL[yr][0].predict_proba(d.loc[R, FEAT_HIST]),
          'B1': predict_b1(b1_tab, R),
          'B0': np.tile(fit_freq(d.loc[fit_rows, 'y'].to_numpy()), (int(R.sum()), 1))}
    if yr == 2026:                                       # the LR exactly as in T4
        prep = fit_prep(FIT, NUM_H, CAT_FEATURES)
        Xf, Xr = transform(prep, FIT), transform(prep, R)
        assert list(Xf.columns) == list(Xr.columns), 'one-hot columns differ'
        lr = fit_lr(Xf.values, d.loc[FIT, 'y'].to_numpy())
        PR['B2-hist +V'] = apply_vector(lr.predict_proba(Xr.values), a26, b26)
    L = {k: per_ball_logloss(v, yv) for k, v in PR.items()}
    for balls, mask in [('all', np.ones(len(yv), bool)), ('death', dth)]:
        if mask.sum() == 0: continue
        base = dict(season=yr, balls=balls, n_balls=int(mask.sum()), matches=len(set(mt[mask])))
        for k in L:
            m_, lo, hi, _, _ = slice_paired(L[k], np.zeros_like(L[k]), mask, mt)
            rows.append({**base, 'what': k, 'value': m_, 'low': lo, 'high': hi})
        for a, b in [('X-hist', 'B1'), ('X-hist', 'B2-hist +V')]:
            if b in L:
                m_, lo, hi, _, _ = slice_paired(L[a], L[b], mask, mt)
                rows.append({**base, 'what': f'{a} - {b}', 'value': m_, 'low': lo, 'high': hi})
rain = pd.DataFrame(rows)
rain.to_csv(DATA + 'rain_scored.csv', index=False)
print(rain.round(4).to_string(index=False))
# for reference, normal 2026 balls: X-hist 1.4787, B1 1.5204, X - B1 -0.0417

 season balls  n_balls  matches                what   value     low    high
   2025   all      698        5              X-hist  1.4458  1.4023  1.5168
   2025   all      698        5                  B1  1.4698  1.4222  1.5495
   2025   all      698        5                  B0  1.4857  1.4412  1.5532
   2025   all      698        5         X-hist - B1 -0.0240 -0.0360 -0.0142
   2025 death      114        3              X-hist  1.5274  1.4574  1.5922
   2025 death      114        3                  B1  1.5110  1.4118  1.5627
   2025 death      114        3                  B0  1.5002  1.3958  1.5608
   2025 death      114        3         X-hist - B1  0.0164 -0.0325  0.0751
   2026   all      382        3              X-hist  1.6382  1.4462  1.8431
   2026   all      382        3                  B1  1.6637  1.3918  1.8573
   2026   all      382        3                  B0  1.6468  1.4174  1.7700
   2026   all      382        3          B2-hist +V  1.6523  1.3925  1.8831
   2026   al

In [10]:
# ===== A6. Temperature check on X-hist (TYPE IT) =====
from scipy.optimize import minimize_scalar

def fit_temperature(P, y):
    """The single T that minimises the log-loss of softmax(log p / T)."""
    Z = to_logits(P)
    nll = lambda T: per_ball_logloss(softmax(Z / T), y).mean()
    return minimize_scalar(nll, bounds=(0.5, 3.0), method='bounded').x

def ece_class(P, y, k, mask, n_bins=10):
    """Ball-weighted |mean predicted - actual share| over 10 equal-count bins, in points."""
    p, hit = P[mask, k], (y[mask] == k).astype(float)
    bins = np.array_split(np.argsort(p, kind='stable'), n_bins)
    return 100 * sum(len(b) * abs(p[b].mean() - hit[b].mean()) for b in bins) / len(p)

y25, y26 = d.loc[VA, 'y'].to_numpy(), d.loc[TE, 'y'].to_numpy()
T = fit_temperature(FULL[2025][2], y25)            # learned on 2025, never on 2026
P_raw = FULL[2026][2]
P_T = softmax(to_logits(P_raw) / T)
print(f'T learned on 2025: {T:.3f}   (>1 softens, <1 sharpens; 2024 gave 1.039)')

mt26, dth26 = MATCH[TE], DEATH[TE]
L_raw, L_T = per_ball_logloss(P_raw, y26), per_ball_logloss(P_T, y26)
SETS = [('all', np.ones(len(y26), bool)), ('death', dth26)]
for balls, mask in SETS:
    m_, lo, hi, _, _ = slice_paired(L_T, L_raw, mask, mt26)
    print(f'{balls:5s} raw {L_raw[mask].mean():.4f} | +T {L_T[mask].mean():.4f} | +T - raw {m_:+.4f} [{lo:+.4f}, {hi:+.4f}]')

rows = []
for balls, mask in SETS:
    for k, cl in enumerate(CLASSES):
        hit = 100 * (y26 == k).astype(float)
        for name, Pm in [('raw', P_raw), ('+T', P_T)]:
            g, lo, hi, _, _ = slice_paired(100 * Pm[:, k], hit, mask, mt26)
            rows.append(dict(balls=balls, cls=cl, model=name, gap_pts=g, low=lo, high=hi,
                             within_error=bool(lo <= 0 <= hi), ece=ece_class(Pm, y26, k, mask)))
cal = pd.DataFrame(rows)
cal.to_csv(DATA + 'temperature_check_2026.csv', index=False)
print(cal.round(2).to_string(index=False))
# check: raw, all balls, dots should reproduce T11's gap of about -0.95 [-1.87, -0.05]

T learned on 2025: 0.971   (>1 softens, <1 sharpens; 2024 gave 1.039)
all   raw 1.4787 | +T 1.4787 | +T - raw -0.0000 [-0.0004, +0.0003]
death raw 1.5680 | +T 1.5678 | +T - raw -0.0002 [-0.0009, +0.0005]
balls    cls model  gap_pts   low  high  within_error  ece
  all      0   raw    -0.95 -1.87 -0.05         False 1.03
  all      0    +T    -0.78 -1.70  0.11          True 0.90
  all      1   raw    -0.21 -1.02  0.60          True 1.09
  all      1    +T     0.44 -0.37  1.25          True 1.09
  all      2   raw     0.12 -0.30  0.54          True 0.46
  all      2    +T    -0.10 -0.52  0.31          True 0.42
  all      3   raw    -0.03 -0.09  0.02          True 0.09
  all      3    +T    -0.06 -0.11  0.00          True 0.09
  all      4   raw     0.59 -0.00  1.20          True 0.81
  all      4    +T     0.42 -0.17  1.04          True 0.66
  all      6   raw     0.15 -0.24  0.58          True 0.67
  all      6    +T    -0.05 -0.44  0.38          True 0.71
  all Wicket   raw     0.34 -

In [11]:
# ===== A7. Block ablation: all in-match context, all history (COPY-PASTE, ~5 min on GPU) =====
saved = np.load(DATA + 'ablation_preds.npz')
CONTEXT = ['delivery', 'chase', 'partnership', 'striker', 'bowler_spell', 'momentum']
BLOCKS = {'all_inmatch_context': sum((G[g] for g in CONTEXT), []),
          'all_history': list(HIST)}
z_bl = norm.ppf(1 - 0.025 / (2 * len(BLOCKS)))       # 2 blocks x (all, death) per season
rows = []
for yr in YEARS:
    sc = SEASON[yr]['score']
    yv, mt, dth = d.loc[sc, 'y'].to_numpy(), MATCH[sc], DEATH[sc]
    L_full = per_ball_logloss(saved[f'{yr}_full'], yv)
    for b, cols in BLOCKS.items():
        _, n, Pb = refit_without(cols, yr)
        L_b = per_ball_logloss(Pb, yv)
        for balls, mask in [('all', np.ones(len(yv), bool)), ('death', dth)]:
            mean, lo, hi, se, bonf = slice_paired(L_b, L_full, mask, mt, z_bonf=z_bl)
            rows.append(dict(season=yr, block=b, n_feats=len(cols), trees=n, balls=balls,
                             loss_if_dropped=mean, low=lo, high=hi, bonferroni=bonf))
blk = pd.DataFrame(rows)
blk['verdict'] = np.select([blk['low'] > 0, blk['high'] < 0], ['block helps', 'block hurts'], "can't tell")
blk.to_csv(DATA + 'ablation_blocks.csv', index=False)
print(f'Bonferroni z = {z_bl:.3f}')
print(blk.round(4).to_string(index=False))

[0]	validation_0-mlogloss:1.54232
[200]	validation_0-mlogloss:1.47724
[377]	validation_0-mlogloss:1.47691
[0]	validation_0-mlogloss:1.54260
[200]	validation_0-mlogloss:1.48305
[342]	validation_0-mlogloss:1.48253
[0]	validation_0-mlogloss:1.53531
[198]	validation_0-mlogloss:1.47540
[0]	validation_0-mlogloss:1.53584
[200]	validation_0-mlogloss:1.48358
[271]	validation_0-mlogloss:1.48359
Bonferroni z = 2.498
 season               block  n_feats  trees balls  loss_if_dropped     low   high  bonferroni     verdict
   2025 all_inmatch_context       39    328   all           0.0039  0.0024 0.0054        True block helps
   2025 all_inmatch_context       39    328 death           0.0058  0.0015 0.0103        True block helps
   2025         all_history       28    293   all           0.0109  0.0081 0.0139        True block helps
   2025         all_history       28    293 death           0.0073  0.0017 0.0132        True block helps
   2026 all_inmatch_context       39    149   all           0